In [1]:
# ==========================================================
# Linear Regression vs Ridge vs Lasso Regression
# Diabetes Dataset
# ==========================================================


# -----------------------------
# Import Libraries
# -----------------------------

import numpy as np
import pandas as pd

from sklearn.datasets import load_diabetes

from sklearn.model_selection import (
    train_test_split,
    GridSearchCV
)

from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import (
    LinearRegression,
    Ridge,
    Lasso
)

from sklearn.metrics import (
    mean_squared_error,
    r2_score
)


# ==========================================================
# 1. LOAD DATASET
# ==========================================================

print("="*60)
print("DIABETES DATASET")
print("="*60)


data = load_diabetes()


X = data.data
y = data.target


print("Number of samples :", X.shape[0])
print("Number of features:", X.shape[1])


# ==========================================================
# 2. TRAIN TEST SPLIT
# ==========================================================


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)


print("\nTraining samples:", X_train.shape[0])
print("Testing samples :", X_test.shape[0])


# ==========================================================
# 3. FEATURE SCALING
# ==========================================================


scaler = StandardScaler()


X_train = scaler.fit_transform(
    X_train
)


X_test = scaler.transform(
    X_test
)


print("\nFeature scaling completed")


# ==========================================================
# 4. STANDARD LINEAR REGRESSION
# ==========================================================


print("\nTraining Linear Regression...")


linear_model = LinearRegression()


linear_model.fit(
    X_train,
    y_train
)


linear_prediction = linear_model.predict(
    X_test
)



# ==========================================================
# 5. RIDGE REGRESSION
# ==========================================================


print("\nTraining Ridge Regression...")


ridge = Ridge()


# Hyperparameter tuning

ridge_params = {

    "alpha":
    [0.01,0.1,1,10,100]

}


ridge_grid = GridSearchCV(
    ridge,
    ridge_params,
    cv=5,
    scoring="neg_mean_squared_error"
)


ridge_grid.fit(
    X_train,
    y_train
)



best_ridge = ridge_grid.best_estimator_


ridge_prediction = best_ridge.predict(
    X_test
)



print(
    "Best Ridge alpha:",
    ridge_grid.best_params_
)



# ==========================================================
# 6. LASSO REGRESSION
# ==========================================================


print("\nTraining Lasso Regression...")


lasso = Lasso(
    max_iter=10000
)



lasso_params = {

    "alpha":
    [0.001,0.01,0.1,1,10]

}



lasso_grid = GridSearchCV(
    lasso,
    lasso_params,
    cv=5,
    scoring="neg_mean_squared_error"
)



lasso_grid.fit(
    X_train,
    y_train
)



best_lasso = lasso_grid.best_estimator_



lasso_prediction = best_lasso.predict(
    X_test
)



print(
    "Best Lasso alpha:",
    lasso_grid.best_params_
)



# ==========================================================
# 7. EVALUATION FUNCTION
# ==========================================================


def evaluate_model(
    name,
    y_true,
    prediction
):

    mse = mean_squared_error(
        y_true,
        prediction
    )


    r2 = r2_score(
        y_true,
        prediction
    )


    print("\n" + "="*50)
    print(name)
    print("="*50)

    print(
        "Mean Squared Error (MSE):",
        round(mse,4)
    )

    print(
        "R-squared Score:",
        round(r2,4)
    )


    return mse,r2



# ==========================================================
# 8. MODEL COMPARISON
# ==========================================================


results = []


models = [

    (
        "Linear Regression",
        linear_prediction
    ),

    (
        "Ridge Regression",
        ridge_prediction
    ),

    (
        "Lasso Regression",
        lasso_prediction
    )

]



for name,pred in models:

    mse,r2 = evaluate_model(
        name,
        y_test,
        pred
    )


    results.append(
        [
            name,
            mse,
            r2
        ]
    )



# ==========================================================
# 9. FINAL RESULTS TABLE
# ==========================================================


print("\n")
print("="*60)
print("FINAL PERFORMANCE COMPARISON")
print("="*60)



comparison = pd.DataFrame(

    results,

    columns=[
        "Model",
        "MSE",
        "R2 Score"
    ]

)



print(
    comparison.round(4)
)



# ==========================================================
# 10. COEFFICIENT COMPARISON
# ==========================================================


print("\n")
print("="*60)
print("MODEL COEFFICIENTS")
print("="*60)



coef_table = pd.DataFrame({

    "Feature":
    data.feature_names,


    "Linear":
    linear_model.coef_,


    "Ridge":
    best_ridge.coef_,


    "Lasso":
    best_lasso.coef_

})


print(
    coef_table.round(4)
)



# ==========================================================
# 11. LASSO FEATURE SELECTION
# ==========================================================


print("\n")
print("="*60)
print("LASSO ZERO COEFFICIENTS")
print("="*60)



zero_features = np.sum(
    best_lasso.coef_ == 0
)


print(
    "Number of removed features:",
    zero_features
)



# ==========================================================
# CONCLUSION
# ==========================================================


print("\n")
print("="*60)
print("SUMMARY")
print("="*60)


print("""
Linear Regression:
- No regularization.
- Can overfit when features are correlated.

Ridge Regression:
- Uses L2 regularization.
- Shrinks coefficients.
- Helps reduce overfitting.

Lasso Regression:
- Uses L1 regularization.
- Can make coefficients exactly zero.
- Performs feature selection.

Best model is selected based on lower MSE
and higher R-squared score.
""")

DIABETES DATASET
Number of samples : 442
Number of features: 10

Training samples: 353
Testing samples : 89

Feature scaling completed

Training Linear Regression...

Training Ridge Regression...
Best Ridge alpha: {'alpha': 10}

Training Lasso Regression...
Best Lasso alpha: {'alpha': 1}

Linear Regression
Mean Squared Error (MSE): 2900.1936
R-squared Score: 0.4526

Ridge Regression
Mean Squared Error (MSE): 2875.7787
R-squared Score: 0.4572

Lasso Regression
Mean Squared Error (MSE): 2824.5681
R-squared Score: 0.4669


FINAL PERFORMANCE COMPARISON
               Model        MSE  R2 Score
0  Linear Regression  2900.1936    0.4526
1   Ridge Regression  2875.7787    0.4572
2   Lasso Regression  2824.5681    0.4669


MODEL COEFFICIENTS
  Feature   Linear    Ridge    Lasso
0     age   1.7538   1.9313   0.6870
1     sex -11.5118 -11.0282  -9.2975
2     bmi  25.6071  25.6056  26.2192
3      bp  16.8289  16.3042  15.6573
4      s1 -44.4489 -12.9342  -8.2282
5      s2  24.6410   0.5382  -0.00